# Satellite Telemetry Simulation

This notebook runs a real-time simulation of satellite cell shifting and visualizes the telemetry data as it changes.

### How it Works:
1.  **Initialization**: Sets up the initial state for 3 satellites, each with 7 cells.
2.  **Simulation Loop**: Enters an infinite loop that can be stopped by interrupting the kernel.
3.  **Cyclic Shift**: Every 90 seconds, it performs a `+1` cyclic left shift on all satellite cell arrays.
4.  **Live Plotting**: After each shift, it clears the cell output and redraws the plot to show the latest telemetry values for the first cell of each satellite.

To run the simulation, execute the code cell below. To stop it, interrupt the kernel (from the menu or by pressing `I, I` in command mode).

In [ ]:
import time
import matplotlib.pyplot as plt
from collections import deque
from IPython.display import clear_output, display

NUM_SATELLITES = 3
NUM_CELLS = 7
SHIFT_INTERVAL_SECONDS = 90  # Telemetry update interval

arrays = [
    deque(range(0, 7)),          # Satellite 1
    deque(range(10, 17)),        # Satellite 2
    deque(range(20, 27)),        # Satellite 3
]

# plotting logs: to track the value of the first cell of each satellite over time.
history = {f"sat_{i+1}_cell_0": [] for i in range(NUM_SATELLITES)}
timestamps = []

def cyclic_shift_and_log():
    for i, arr in enumerate(arrays):
        # value at index 0 is what we want to log before it moves
        pre_shift_first_val = arr[0]
        history[f"sat_{i+1}_cell_0"].append(pre_shift_first_val)

        # cyclic shift
        arr.rotate(-1)  # rotates left

    timestamps.append(time.time())

def display_telemetry_plot():
    fig, ax = plt.subplots(figsize=(12, 7))
    plt.style.use('seaborn-v0_8-darkgrid')

    start_time = timestamps[0]
    relative_times = [(t - start_time) for t in timestamps]

    for i in range(NUM_SATELLITES):
        sat_key = f"sat_{i+1}_cell_0"
        #get satellite index and plot its first cell value over time
        #store the first cell value history for this satellite
        #update the label for the plot
        cell_history = history[sat_key]
        cell_value = arrays[i][0]
        ax.plot(relative_times, cell_history, marker='o', linestyle='--', label=f'Satellite {i+1} (Initial Value: {cell_value})')

    ax.set_title('Satellite Cell Shifts Over Time', fontsize=16)
    ax.set_xlabel(f'Time (seconds) - Updates every {SHIFT_INTERVAL_SECONDS}s', fontsize=12)
    ax.set_ylabel('Cell Value', fontsize=12)
    ax.legend()
    ax.grid(True)
    
    # plot in the notebook or colab
    # https://colab.research.google.com/drive/1oO4PaPrfZVrxoy5wmVjOtoPREKtJszkG#scrollTo=0MMDDRGxAzI5
    display(fig)
    plt.close(fig) # Close the figure to free memory

print("Starting satellite telemetry simulation...")
print(f"A cyclic shift will occur every {SHIFT_INTERVAL_SECONDS} seconds.")
print("To stop, interrupt the kernel.")

try:
    while True:
        # call the log function
        cyclic_shift_and_log()
        
        # Clear data to display new
        clear_output(wait=True)
        display_telemetry_plot()
        
        print(f"--- {time.strftime('%Y-%m-%d %H:%M:%S')} ---")
        for i, arr in enumerate(arrays):
            print(f"Satellite {i+1} new state: {list(arr)}")
        print(f"\nWaiting for {SHIFT_INTERVAL_SECONDS} seconds until the next shift...")
        
        # 90-second interval
        time.sleep(SHIFT_INTERVAL_SECONDS)
        
except KeyboardInterrupt:
    clear_output(wait=True)
    print("Simulation stopped by user.")
    print("Final telemetry data plot:")
    display_telemetry_plot() # updated plot